In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split

iris = datasets.load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
x = df.loc[:, ["petal length (cm)", "petal width (cm)"]]
x_train, x_test, y_train, y_test = train_test_split(x, iris.target, test_size=0.25, random_state=0)

x_train = x_train.to_numpy()


n_qubits = 5
num_class = 3

In [2]:
from scikit_quri.circuit import create_qcl_ansatz

circuit = create_qcl_ansatz(n_qubits, 3, 1.0)

theta = np.array([np.random.random() for _ in range(circuit.learning_params_count)])
# circuitとparamsをscaluqへ
(circuit, params) = circuit.to_batched(x_train, theta)
print(params.shape)

(112, 55)


In [3]:
from collections.abc import Sequence, Callable
def batched_param_mapper(
    param_mapper: Callable[[Sequence[float]], dict[str, list[float]]],
    params_list: Sequence[Sequence[float]],
) -> dict[str, list[float]]:
    mapped = [param_mapper(params) for params in params_list]
    keys = mapped[0].keys()
    return {k: [m[k][0] for m in mapped] for k in keys}

In [4]:
from scaluq.default.f64 import StateVectorBatched
from quri_parts.core.operator import Operator, pauli_label
from quri_parts_scaluq.estimator import estimate

operators = [Operator({pauli_label(f"Z {i}"): 1.0}) for i in range(num_class)]

state_vector = StateVectorBatched(len(params), n_qubits)
state_vector.set_zero_state()

results = estimate(state_vector, circuit, operators, params)
print(len(results), len(results[0]))  # (n_operators, n_samples)

[Info] Library 'scaluq' is using backend: scaluq.default.f64
3 112


Kokkos::OpenMP::initialize WARNING: OMP_PROC_BIND environment variable not set
  In general, for best performance with OpenMP 4.0 or better set OMP_PROC_BIND=spread and OMP_PLACES=threads
  For best performance with OpenMP 3.1 set OMP_PROC_BIND=true
  For unit testing set OMP_PROC_BIND=false

